# How2Sign Holistic -> PoseT5 Pretrain
Builds a manifest from the public Kaggle How2Sign Holistic dataset and pretrains the current PoseT5 lane on cloud GPU.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

def query_gpu_info() -> tuple[str, float]:
    result = subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,compute_cap',
        '--format=csv,noheader',
    ], text=True).strip().splitlines()[0]
    name, capability = [part.strip() for part in result.split(',', 1)]
    return name, float(capability)

gpu_name, gpu_capability = query_gpu_info()
print({'gpu_name': gpu_name, 'gpu_capability': gpu_capability})
if gpu_capability < 7.0:
    compatible_candidates = [
        ('2.2.0', '0.17.0', '2.2.0'),
        ('2.2.1', '0.17.1', '2.2.1'),
        ('2.2.2', '0.17.2', '2.2.2'),
        ('2.3.0', '0.18.0', '2.3.0'),
        ('2.3.1', '0.18.1', '2.3.1'),
        ('2.4.0', '0.19.0', '2.4.0'),
        ('2.4.1', '0.19.1', '2.4.1'),
        ('2.5.0', '0.20.0', '2.5.0'),
        ('2.5.1', '0.20.1', '2.5.1'),
    ]
    selected = None
    for torch_version, torchvision_version, torchaudio_version in compatible_candidates:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '--force-reinstall',
            f'torch=={torch_version}',
            f'torchvision=={torchvision_version}',
            f'torchaudio=={torchaudio_version}',
            '--index-url',
            'https://download.pytorch.org/whl/cu121',
        ])
        arch_probe = subprocess.check_output([
            sys.executable,
            '-c',
            'import json, torch; print(json.dumps(sorted(torch.cuda.get_arch_list())))',
        ], text=True).strip()
        print({'candidate_torch': torch_version, 'arch_probe': arch_probe})
        if 'sm_60' in arch_probe:
            selected = (torch_version, torchvision_version, torchaudio_version)
            break
    if selected is None:
        raise RuntimeError('No available cu121 torch wheel in Kaggle exposed sm_60 support for P100.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy<2'])

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable in this Kaggle session.')
arch_list = sorted(torch.cuda.get_arch_list())
runtime_gpu_name = torch.cuda.get_device_name(0)
runtime_gpu_capability = torch.cuda.get_device_capability(0)
print({
    'torch_version': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'runtime_gpu_name': runtime_gpu_name,
    'runtime_gpu_capability': runtime_gpu_capability,
    'arch_list': arch_list,
})
if runtime_gpu_capability[0] < 7 and 'sm_60' not in arch_list:
    raise RuntimeError(
        f'Installed torch {torch.__version__} still lacks sm_60 support for {runtime_gpu_name}: {arch_list}'
    )

def find_first_file(pattern: str) -> Path:
    for path in sorted(INPUT_ROOT.rglob(pattern)):
        if path.is_file():
            return path
    raise FileNotFoundError(f'Could not find file matching {pattern!r} under {INPUT_ROOT}')

def find_first_dir(name: str) -> Path:
    for path in sorted(INPUT_ROOT.rglob(name)):
        if path.is_dir():
            return path
    raise FileNotFoundError(f'Could not find directory named {name!r} under {INPUT_ROOT}')

def find_code_root() -> tuple[Path | None, Path]:
    repo_bundle = next((path for path in sorted(INPUT_ROOT.rglob('repo_bundle.zip')) if path.is_file()), None)
    if repo_bundle is not None:
        return repo_bundle, repo_bundle.parent
    train_script = find_first_file('kaggle_train_pose_t5.py')
    return None, train_script.parent.parent

repo_bundle, code_root = find_code_root()
CODE_DATASET = str(code_root)
HOW2SIGN_DATASET = str(find_first_dir('how2sign_holistic_features').parent)
MT5_LOCAL = str(find_first_file('spiece.model').parent)
REPO_DIR = Path('/tmp/thai_sign_code_h2s')
MANIFEST_DIR = Path('/kaggle/working/how2sign_holistic_manifest')
OUT_DIR = Path('/kaggle/working/pose_t5_how2sign_pretrain')
shutil.rmtree(REPO_DIR, ignore_errors=True)
REPO_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

if repo_bundle is not None:
    with zipfile.ZipFile(repo_bundle) as archive:
        archive.extractall(REPO_DIR)
else:
    shutil.copytree(code_root, REPO_DIR, dirs_exist_ok=True)

for package in ['numpy<2', 'transformers>=4.40', 'sentencepiece>=0.2.0', 'sacrebleu>=2.4.0', 'portalocker>=2.0', 'pandas>=2.0']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')

subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts' / 'build_how2sign_holistic_manifest.py'),
    '--input-root', HOW2SIGN_DATASET,
    '--out-dir', str(MANIFEST_DIR),
], env=env)

subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts' / 'kaggle_train_pose_t5.py'),
    '--data-roots', str(MANIFEST_DIR),
    '--out-dir', str(OUT_DIR),
    '--base-model', MT5_LOCAL,
    '--epochs', '2',
    '--batch-size', '4',
    '--grad-accum', '4',
    '--lr', '1e-4',
    '--dropout', '0.1',
    '--weight-decay', '0.01',
    '--max-runtime-min', '690',
    '--eval-steps', '20',
    '--checkpoint-steps', '20',
    '--early-stopping-patience', '0',
    '--early-stopping-metric', 'val_chrf',
    '--num-workers', '2',
    '--preload-train-features', 'false',
    '--balance-sources', 'false',
    '--required-sources', 'how2sign',
    '--manifest-quality-sources', 'how2sign',
    '--fail-on-manifest-quality', 'false',
    '--allow-noop-resume', 'false',
    '--split-policy', 'manifest',
    '--evaluate-after-train', 'false',
], env=env)

print(sorted(path.name for path in OUT_DIR.iterdir()))
